# 03 行列分解 — 評価の「点数」を使うと、個人化が効く

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 好みを少数の軸で表す行列分解（ALS・BPR）を試す回です。
> - **たとえ話**: 好みを「恋愛度」「アクション度」といった軸の数字で表し、掛け算で相性を出す。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-5（行列分解）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> 🔒 すべて**合成データ**です。⏱ 所要時間: 説明 15分 ＋ 実行 約45秒

## ゴール
- **行列分解**（潜在因子モデル）の仕組みを理解し、自作の **ALS**（評価の点数を当てる）と **BPR**（順位を直接学ぶ）を動かす。
- 02 の近傍法が及ばなかった**個人化**が、**評価の点数を使う**と効くことを確かめる。
- **BPR**が、**露出の偏り**（人気）を、そのまま学んでしまうことを知る。
- 学習した**因子が、真の因子に近いか**（合成データだから確かめられる）、**データが増えるとどうなるか**を見る。

## 前のNotebookとのつながり
02 の近傍法は、**評価した／しない**（または好き／そうでない）という**0/1**だけを使い、**平均評価順（個人化なし）にも及びません**でした。
ここでは、評価の**点数（1〜5）**を使います。

## 行列分解とは
ユーザーと商品を、少数（k個）の**潜在因子**（好みの軸・商品の特徴の軸）で表します。

```
予測 = 全体の平均 + ユーザーの甘辛 + 商品の質 + ユーザーの因子 · 商品の因子
```

**因子の内積が大きい**ほど、「このユーザーは、この商品を好む」と予測します。観測された評価に、なるべく合うように、因子を学習します。

- **ALS**: **点数を、二乗誤差で当てにいく**。ユーザーの因子を固定して商品を解き、次に逆、を繰り返す。
- **BPR**: **順位を直接**学ぶ。「好きな商品」が、「まだ知らない商品」より**上に来る**ように、因子を調整する。

## キーワード
- **潜在因子**: データから学ぶ、目に見えない特徴の軸。**k（因子の数）**が大きいほど、複雑な好みを表せる。
- **正則化（reg）**: 因子を小さく保つ（単純にする）力。**弱すぎると過学習**（学習データに合いすぎ）、**強すぎると、因子が使われない**。
- **過学習**: 学習データには合うが、新しいデータには合わない。
- **外し役（負例）**: BPR で、「好きな商品」の比較相手に選ぶ「知らない商品」。**選び方**で、学ぶものが変わる。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import run_phase3

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

e1 = run_phase3.e1()
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, col, title, cmap in [(ax[0], "RMSE", "検証のRMSE（小さいほど良い）", "viridis_r"), (ax[1], "観測NDCG", "観測NDCG（大きいほど良い）", "viridis"),
                            (ax[2], "本当のprecision", "本当のprecision（大きいほど良い）", "viridis")]:
    piv = e1.pivot(index="k", columns="reg", values=col)
    im = a.imshow(piv.values, cmap=cmap, aspect="auto")
    a.set_xticks(range(len(piv.columns))); a.set_xticklabels(piv.columns); a.set_yticks(range(len(piv.index))); a.set_yticklabels(piv.index)
    a.set(title=title, xlabel="正則化 reg（右ほど強い）", ylabel="因子の数 k")
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            a.text(j, i, f"{piv.values[i, j]:.2f}" if col != "観測NDCG" else f"{piv.values[i, j]:.3f}", ha="center", va="center", color="w", fontsize=7)
fig.tight_layout()

## E1. 因子の数と正則化

**何をしているか**: ALSの**因子の数 k**（1〜32）と**正則化 reg**（0.5〜20）を変えて、(1) 検証データのRMSE、(2) 観測NDCG、(3) 本当のprecision を、格子（ヒートマップ）で見ます（1つの世界）。

**出力の見方**:
- **RMSE（左）は、「ちょうど良い」ところが最小**: k=4・reg=5 で **0.762**（Phase1 のバイアスモデルは0.878）。
- **過学習（左下）**: regが弱く（0.5）、kが大きい（16）と、RMSEは **1.04** まで悪化します。因子が、学習データのノイズに合いすぎるためです。
- **正則化が強すぎる（reg=20。右端）**: どのkでも、RMSEは **0.865** で同じ。因子が使われず、**バイアスモデル（0.878）とほぼ同じ**になります。
- **本当のprecision（右）**は、RMSEと**よく対応**します。最良は k=8・reg=5 の **0.787**（k=4・reg=5 は 0.786）。regが弱い過学習の領域では、**0.55〜0.58**まで落ちます。
- **観測NDCG（中）**も、同じ設定（k=4〜8・reg=5）で最大（**0.105**）。ただし、**人気順（0.198）の約半分**です。
- ここでは、**RMSE・観測NDCG・本当のprecisionの最良の設定が、ほぼ一致**します（近傍法のときは、観測で選ぶと本当が下がりました）。

In [ ]:
e2 = run_phase3.e2()
e2.set_index("手法")

## E2. 手法の比較（世界10回の平均±標準誤差）

**何をしているか**: これまでの手法と、ALS・BPR（外し役の選び方を2通り）を比べます。ALSは k=4・reg=5、BPRは k=16・学習率0.05・40エポックで、固定です。

**出力の見方**:
- **ALS（評価の点数・k=4）は、本当の好みで、最良**: precision **0.798（±0.010）**。平均評価順（0.616）、近傍法（0.495）、人気順（0.305）を、**大きく上回ります**。**個人化が、初めて効きました**。カバレッジは0.23。
- ところが、**観測NDCGは、0.109**で、**人気順（0.168）や近傍法（0.180）より低い**。**観測テストだけで評価すると、ALSは「人気順より悪い」と判断してしまいます**。本当の好みでは、**人気順の約2.6倍**（0.798 対 0.305）の当たり。
- **BPR（外し役：一様）**: 観測NDCG **0.160**（人気順並み）、本当のprecision **0.473**。**露出（人気）を、そのまま学んでいます**。好きな商品（人気に偏る）を、一様に選んだ知らない商品より上に置くので、**人気の商品の点数が上がる**のです。
- **BPR（外し役：人気に比例）**: 観測NDCG **0.054**（観測では、ほぼ最下位）、本当のprecision **0.519**、カバレッジ **0.72**。**外し役に、人気の商品を、わざと選ぶ**と、「人気だから好き」という近道が使えなくなり、**偏りが弱まります**。観測では下がり、本当では上がる。
- **上限**は0.973。ALSは、まだ0.18足りません。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
names = list(e2["手法"])
obs = [float(x.split(" ")[0]) for x in e2["観測NDCG@10"]]
tru = [float(x.split(" ")[0]) for x in e2["本当のprecision@10"]]
x = np.arange(len(names))
ax.bar(x - 0.2, np.array(obs) / max(obs), 0.4, label="観測NDCG（最大を1にそろえる）")
ax.bar(x + 0.2, tru, 0.4, label="本当のprecision")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha="right", fontsize=8); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
ax.set(title="観測テストと、本当の好みで、評価が違う")
fig.tight_layout()

**出力の見方**: 青（観測）と橙（本当）が、**逆の高さ**になる手法があります。**人気順・近傍法・BPR（一様）**は、観測が高く本当が低い。**ALS・BPR（人気に比例）・平均評価順**は、その逆。**観測テストだけで選ぶと、本当に良い手法（ALS）を、見落とします**。

In [ ]:
e3 = run_phase3.e3()
display(e3.set_index("ユーザーあたりの評価数（目安）"))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(e3["ユーザーあたりの評価数（目安）"], e3["ALSの本当のprecision"], "o-", label="ALS")
ax[0].plot(e3["ユーザーあたりの評価数（目安）"], e3["平均評価順の本当のprecision"], "s-", label="平均評価順（個人化なし）")
ax[0].set(xscale="log", title="データが増えると、個人化が効く", xlabel="ユーザーあたりの評価数（目安）", ylabel="本当のprecision@10"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3, which="both")
ax[1].plot(e3["ユーザーあたりの評価数（目安）"], e3["商品因子の重なり(上位4次元)"], "o-", label="商品の因子")
ax[1].plot(e3["ユーザーあたりの評価数（目安）"], e3["ユーザー因子の重なり(上位4次元)"], "s-", label="ユーザーの因子")
ax[1].set(xscale="log", ylim=(0, 1), title="学習した因子と、真の因子の重なり（1＝一致）", xlabel="ユーザーあたりの評価数（目安）"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3, which="both")
fig.tight_layout()

## E3. 因子は、真の因子に近いか。データが増えると？

**何をしているか**: データを作るときの**評価の数**（ユーザーあたり10・25・50・100件の目安）を変えて、ALS（k=4・reg=5）の成績と、**学習した因子が、真の因子（上位4次元）とどれだけ重なるか**を測ります。合成データだから、**真の因子**（本当の好みの行列から、SVDで取り出した）が分かるので、確かめられます。重なりは、1が完全に一致、0が無関係。

**出力の見方**:
- **評価が少ない（10件/人）と、個人化の効果は小さい**: ALSの本当のprecision **0.682**、平均評価順 0.632（差0.05）。因子の重なりも **0.21〜0.24**（ほとんど復元できない）。
- **評価が増えるほど、ALSが伸びる**: 25件 **0.786**、50件 **0.869**、100件 **0.894**。平均評価順は **0.63〜0.67**で、ほとんど変わりません。**データが増えるほど、個人化の価値が増える**。
- **因子の重なり**も、**0.21 → 0.51 → 0.73 → 0.88**（商品の因子）と上がります。**ユーザーの好みの軸を、データから復元している**ことが、確かめられました。
- 評価の誤差（RMSE）も、全部の組・本当の好みで **0.920 → 0.482** に減ります。
- Phase2 で、**近傍法は、評価が多いユーザーでも良くならない**ことを見ました。**行列分解は、データが増えると、確実に良くなる**。**評価の点数（強さ）を使う**ことの価値です。
- 注意: 標準の設定（25件/人）では、因子の重なりは **0.5**程度で、**半分しか復元できていません**。それでも、**順位の当たりは、平均評価順を大きく上回ります**。

In [ ]:
e4 = run_phase3.e4()
e4.set_index("選び方")

## E4. ハイパーパラメータを、何で選ぶか

**何をしているか**: kとregの格子を、1つの世界で、(1) 検証のRMSE、(2) 観測NDCG、(3) 本当のprecision、のそれぞれで選び、**別の9つの世界**で評価します（Phase2 の近傍法と同じ方法）。

**出力の見方**:
- **どの基準で選んでも、ほぼ同じ設定**です（RMSE・観測NDCGは k=4・reg=5、本当は k=8・reg=5）。別の世界での本当のprecisionは、**0.800 対 0.793**（差0.007）。**差は、誤差の範囲**。
- Phase2 の近傍法では、**観測で選ぶと、本当が約0.03下がりました**。ALSでは、**そうなりません**。理由: ALSは、**点数を当てる**ので、**RMSEと本当の良さが、よく一致**します。近傍法の「α」のように、**人気の効きを変えるつまみ**が、ALSにはないためです。
- 結論: **何で選ぶかが重要かどうかは、手法による**。選んだ設定が、**偏りに引きずられていないか**は、手法ごとに確かめる。

## 試してみよう（演習）
1. **因子の数を変える**: `run_phase3.ALS_DEFAULT = dict(k=4, reg=5.0)` の `k` を `1` と `16` にして、E2のALSの行を見てください（`e2()` を実行）。分かるのは、因子が少なすぎると表せず、多すぎると（regが適切なら）あまり変わらないことです。
2. **正則化を変える**: `reg` を `1` と `20` にしてください。分かるのは、弱すぎる（過学習）／強すぎる（因子が使われない）と、どちらも悪いことです。
3. **BPR の外し役**: `bpr_fit(..., neg_pop_power=0.5)` にして、観測NDCGと本当のprecisionの、中間の結果を確かめてください。分かるのは、人気への偏りを、つまみで調整できることです。
4. **データを増やす**: E3の `events=(10, 25, 50, 100)` を `(25, 200)` にして、評価が非常に多いときの、ALSの成績を見てください。
5. **学習率を変える**: `BPR_DEFAULT` の `lr=0.05` を `0.2` と `0.01` にして、収束の速さと成績の変化を見てください。

## まとめ
- **行列分解（ALS）**は、評価の**点数**を使い、個人化が**効く**: 本当のprecision **0.798**（平均評価順 0.616、近傍法 0.495、人気順 0.305）。
- ところが、**観測NDCGは、人気順より低い**（0.109 対 0.168）。**観測テストだけで評価すると、本当に良い手法を、見落とします**。
- **BPR**（順位を直接学ぶ）は、**外し役を一様に選ぶと、露出（人気）を学ぶ**。**人気に比例して選ぶ**と、偏りが弱まる（観測0.160→0.054、本当0.473→0.519）。
- **データが増えるほど、個人化が効く**（10件/人で+0.05、100件/人で+0.23）。**因子は、真の因子に近づく**（重なり0.21→0.88）。
- **正則化**が重要。弱いと過学習（RMSE 1.04）、強いと因子が使われない（バイアスモデルと同じ）。
- **ハイパーパラメータの選び方**: ALSでは、RMSE・観測・本当で、ほぼ同じ設定が選ばれる（手法による）。

次の Phase4 では、**コールドスタート**（新しいユーザー・商品）と、**コンテンツ（ジャンル）**の併用を調べます。